# LeetCode 3957 — Maximum Sum of M Non-Overlapping Subarrays II bằng Python

- Link: https://leetcode.com/problems/maximum-sum-of-m-non-overlapping-subarrays-ii/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động, hàng đợi đơn điệu, **Aliens trick** (WQS binary search / tối ưu lồi theo số đoạn), tổng tiền tố

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maximumSum(self, nums: list[int], m: int, l: int, r: int) -> int:
```

> Bài này là phiên bản **II** của LeetCode 3956 (cùng đề, khác ràng buộc: `n ≤ 10^5`, `|nums[i]| ≤ 10^5`). Notebook này **độc lập**: tự phát biểu lại đề và tự xây lời giải; nếu bạn đã biết DP `O(n·m)` của bài I thì phần 5–7.2 sẽ là ôn lại ngắn gọn.

## 1. Giới thiệu & mục tiêu học

Cho mảng `nums` độ dài `n ≤ 10^5` và ba số `m`, `l`, `r`. Chọn **ít nhất một và nhiều nhất `m`** mảng con không giao nhau, mỗi mảng con dài trong `[l, r]`, sao cho tổng các phần tử được chọn lớn nhất.

DP quen thuộc `f[t][i]` tốn `O(n · m)` — với `n = m = 10^5` là `10^{10}` bước, quá chậm (hàng chục phút ngay cả khi mỗi bước chỉ vài trăm nano-giây). Ta cần phá chiều "số đoạn `m`". Ý tưởng: nếu hàm `f(k)` = "tổng lớn nhất khi chọn **đúng** `k` đoạn" là **lõm (concave)** theo `k`, ta có thể thay ràng buộc cứng "đúng `m` đoạn" bằng một **khoản phạt `λ` cho mỗi đoạn**, rồi dùng tìm kiếm nhị phân trên `λ`. Mỗi giá trị `λ` chỉ cần một lượt DP `O(n)` (không còn chiều `t`). Đây là **Aliens trick** (còn gọi WQS binary search, từ bài *Aliens* ở IOI 2016).

Sau notebook này bạn sẽ:

1. Hiểu hàm `f(k)` và vì sao tính lõm của nó cho phép đổi "ràng buộc số đoạn" thành "phạt `λ` mỗi đoạn".
2. Viết DP một chiều `h[i]` với phạt `λ`, lấy tối đa trong cửa sổ `[i−r, i−l]` bằng hàng đợi đơn điệu, và **theo dõi số đoạn** khi có nhiều phương án bằng giá trị.
3. Xử lý điểm khó của Aliens: **hòa** (nhiều số đoạn cùng tối ưu) bằng quy tắc "ít đoạn nhất" và cách khôi phục `f(m)` bằng `g(λ) + λ·m`.
4. Viết 4 lời giải (vét cạn, DP `O(n·m)`, Aliens bản dễ hiểu bằng tuple, Aliens bản tối ưu mã hóa khóa số nguyên) và đối chiếu bằng test ngẫu nhiên.
5. Phân tích `O(n · log V)` với `V ≈ 10^{10}` và biết các điều kiện để dùng Aliens.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng `nums` độ dài `n` và ba số nguyên `m`, `l`, `r`. Chọn từ `1` đến `m` mảng con liên tiếp **không chồng lấn**, mỗi mảng con có độ dài trong `[l, r]` (tính cả hai đầu). Trả về tổng lớn nhất của tất cả phần tử thuộc các mảng con được chọn.

**Ví dụ 1:** `nums = [4,1,-5,2]`, `m = 2`, `l = 1`, `r = 3` → `7` (`[4,1]` và `[2]`).

**Ví dụ 2:** `nums = [1,0,3,4]`, `m = 2`, `l = 1`, `r = 2` → `8` (`[1]` và `[3,4]`).

**Ví dụ 3:** `nums = [-1,7,-4]`, `m = 1`, `l = 2`, `r = 3` → `6` (`[-1,7]`).

**Ví dụ 4:** `nums = [-3,-4,-1]`, `m = 2`, `l = 1`, `r = 2` → `-1` (toàn âm, phải chọn ít nhất một đoạn nên chọn `[-1]`).

**Ràng buộc**

- `1 <= n == nums.length <= 10^5`
- `-10^5 <= nums[i] <= 10^5`
- `1 <= m <= n`, `1 <= l <= r <= n`

Lưu ý: (1) **ít nhất một** đoạn nên đáp án có thể âm; (2) hai đoạn được phép **kề nhau**; (3) tổng lớn nhất có thể là `10^5 × 10^5 = 10^{10}` — tràn `int32` nhưng Python không lo.

In [1]:
examples = [
    (dict(nums=[4, 1, -5, 2], m=2, l=1, r=3), 7),
    (dict(nums=[1, 0, 3, 4], m=2, l=1, r=2), 8),
    (dict(nums=[-1, 7, -4], m=1, l=2, r=3), 6),
    (dict(nums=[-3, -4, -1], m=2, l=1, r=2), -1),
]
for args, expected in examples:
    print("nums =", args["nums"], "| m, l, r =", (args["m"], args["l"], args["r"]), "| mong đợi:", expected)

nums = [4, 1, -5, 2] | m, l, r = (2, 1, 3) | mong đợi: 7
nums = [1, 0, 3, 4] | m, l, r = (2, 1, 2) | mong đợi: 8
nums = [-1, 7, -4] | m, l, r = (1, 2, 3) | mong đợi: 6
nums = [-3, -4, -1] | m, l, r = (2, 1, 2) | mong đợi: -1


## 3. Kiến thức nền

### 3.1. Tổng tiền tố và đoạn nửa mở `[j, i)`

`P[0] = 0`, `P[i] = nums[0] + ... + nums[i−1]`. Đoạn chiếm vị trí `j, ..., i−1` có tổng `P[i] − P[j]` và độ dài `i − j`.

In [2]:
nums = [4, 1, -5, 2]
P = [0]
for x in nums:
    P.append(P[-1] + x)
print("P =", P)
print("đoạn [1,3) = nums[1:3] có tổng", P[3] - P[1], "độ dài", 3 - 1)

P = [0, 4, 5, 0, 2]
đoạn [1,3) = nums[1:3] có tổng -4 độ dài 2


### 3.2. Hàm `f(k)` — tổng lớn nhất khi chọn đúng `k` đoạn

Gọi `f(k)` (với `k = 0, 1, ..., K`, `K = n // l` là số đoạn tối đa xếp được) là tổng lớn nhất khi chọn **đúng `k`** đoạn không giao nhau, mỗi đoạn dài `[l, r]`; `f(0) = 0`. Đáp án của đề là `max_{1 ≤ k ≤ m} f(k)`. Ta tính `f` bằng DP chậm (đủ dùng cho mảng nhỏ) để quan sát hình dạng của nó.

In [3]:
NEG = float("-inf")

def exact_values(nums, l, r):
    """f(k) cho k = 0..K bằng DP O(n*K*(r-l+1)) (chỉ để quan sát)."""
    n = len(nums)
    P = [0]
    for x in nums:
        P.append(P[-1] + x)
    Kmax = n // l
    f = [[NEG] * (n + 1) for _ in range(Kmax + 1)]
    for i in range(n + 1):
        f[0][i] = 0
    for t in range(1, Kmax + 1):
        for i in range(1, n + 1):
            best = f[t][i - 1]
            for ln in range(l, min(r, i) + 1):
                if f[t - 1][i - ln] > NEG:
                    best = max(best, f[t - 1][i - ln] + P[i] - P[i - ln])
            f[t][i] = best
    return [f[t][n] for t in range(Kmax + 1)]

vals = exact_values([4, 1, -5, 2, 6, -3, 3, 1], 1, 3)
print("f(k):", vals)
print("độ dốc f(k)-f(k-1):", [vals[k] - vals[k - 1] for k in range(1, len(vals))])

f(k): [0, 8, 13, 17, 17, 17, 17, 14, 9]
độ dốc f(k)-f(k-1): [8, 5, 4, 0, 0, 0, -3, -5]


### 3.3. Hàm lõm (concave) và độ dốc không tăng

`f` **lõm** nếu độ dốc `d_k = f(k) − f(k−1)` **không tăng** khi `k` tăng: `d_1 ≥ d_2 ≥ ... `. Nhìn mảng độ dốc ở trên: các số giảm dần. Ý nghĩa: "đoạn thứ `k` thêm vào" mang lại lợi ích ít hơn đoạn trước — quy luật lợi ích cận biên giảm dần.

Với bài này ta **dùng tính lõm** như một sự thật (đã được kiểm chứng thực nghiệm trên hàng chục nghìn input ở phần 12), và nhấn mạnh rằng đây là điều kiện bắt buộc để Aliens đúng.

In [4]:
import random
rng = random.Random(0)
violations = 0
for _ in range(3000):
    n = rng.randint(1, 12)
    l = rng.randint(1, min(n, 4)); r = rng.randint(l, n)
    nums = [rng.randint(-6, 8) for _ in range(n)]
    v = exact_values(nums, l, r)
    d = [v[k] - v[k - 1] for k in range(1, len(v))]
    if any(d[k] < d[k + 1] for k in range(len(d) - 1)):
        violations += 1
print("số input vi phạm tính lõm trong 3000 input ngẫu nhiên:", violations)

số input vi phạm tính lõm trong 3000 input ngẫu nhiên: 0


### 3.4. Phạt `λ` mỗi đoạn và hàm `g(λ)`

Với một số `λ`, xét bài toán "chọn **bao nhiêu đoạn tùy ý** (kể cả 0), mỗi đoạn bị trừ `λ`": 

`g(λ) = max_k ( f(k) − λ·k )`.

Nếu `f` lõm thì `f(k) − λk` cũng lõm theo `k`: nó **tăng khi `d_k > λ`** và không tăng khi `d_k ≤ λ`. Do đó số đoạn tối ưu là số độ dốc lớn hơn `λ`; λ lớn → ít đoạn; λ nhỏ → nhiều đoạn. Mỗi `g(λ)` tính được bằng **một** lượt DP `O(n)` không có chiều `t`.

In [5]:
vals = exact_values([4, 1, -5, 2, 6, -3, 3, 1], 1, 3)
d = [vals[k] - vals[k - 1] for k in range(1, len(vals))]
print("f(k)      :", vals)
print("độ dốc d_k:", d)
for lam in [0, 1, 2, 3, 4, 5, 6, 7, 8]:
    scores = [vals[k] - lam * k for k in range(len(vals))]
    g = max(scores)
    ks = [k for k, s in enumerate(scores) if s == g]
    print(f"λ={lam}: g(λ)={g}, số đoạn tối ưu ∈ {ks}, #(d_k>λ) = {sum(1 for x in d if x > lam)}")

f(k)      : [0, 8, 13, 17, 17, 17, 17, 14, 9]
độ dốc d_k: [8, 5, 4, 0, 0, 0, -3, -5]
λ=0: g(λ)=17, số đoạn tối ưu ∈ [3, 4, 5, 6], #(d_k>λ) = 3
λ=1: g(λ)=14, số đoạn tối ưu ∈ [3], #(d_k>λ) = 3
λ=2: g(λ)=11, số đoạn tối ưu ∈ [3], #(d_k>λ) = 3
λ=3: g(λ)=8, số đoạn tối ưu ∈ [3], #(d_k>λ) = 3
λ=4: g(λ)=5, số đoạn tối ưu ∈ [2, 3], #(d_k>λ) = 2
λ=5: g(λ)=3, số đoạn tối ưu ∈ [1, 2], #(d_k>λ) = 1
λ=6: g(λ)=2, số đoạn tối ưu ∈ [1], #(d_k>λ) = 1
λ=7: g(λ)=1, số đoạn tối ưu ∈ [1], #(d_k>λ) = 1
λ=8: g(λ)=0, số đoạn tối ưu ∈ [0, 1], #(d_k>λ) = 0


Bạn thấy: số đoạn tối ưu nhỏ nhất bằng `#{k : d_k > λ}` (đếm độ dốc **lớn hơn hẳn** `λ`), số đoạn tối ưu lớn nhất bằng `#{k : d_k ≥ λ}`; khi `λ` đúng bằng một độ dốc thì có **nhiều** số đoạn cùng tối ưu (hòa). Đây là chỗ Aliens dễ sai nếu không có quy tắc phá hòa.

### 3.5. Hàng đợi đơn điệu (sliding window maximum)

Trong DP một chiều ta cần `max_{j ∈ [i−r, i−l]} (h[j] − P[j])`. Dùng `collections.deque` lưu các cặp `(giá trị, chỉ số)` theo giá trị giảm dần: thêm phần tử mới thì bỏ các phần tử cuối nhỏ hơn hoặc bằng; bỏ phần tử đầu quá hạn; phần tử đầu là max.

In [6]:
from collections import deque

a = [3, 1, 4, 1, 5, 9, 2, 6]
w = 3                               # max của cửa sổ độ dài 3 kết thúc tại mỗi vị trí
dq = deque()
out = []
for i, x in enumerate(a):
    while dq and dq[-1][0] <= x:
        dq.pop()
    dq.append((x, i))
    while dq[0][1] <= i - w:
        dq.popleft()
    out.append(dq[0][0])
print(out)                           # [3, 3, 4, 4, 5, 9, 9, 9]
assert out == [max(a[max(0, i - w + 1):i + 1]) for i in range(len(a))]

[3, 3, 4, 4, 5, 9, 9, 9]


### 3.6. Tìm kiếm nhị phân trên số nguyên

Để tìm `λ` nhỏ nhất thỏa một điều kiện đơn điệu (sai ... sai đúng ... đúng), dùng:

```python
lo, hi = nhỏ_nhất_có_thể, giá_trị_chắc_chắn_đúng
while lo < hi:
    mid = (lo + hi) // 2
    if đúng(mid): hi = mid
    else: lo = mid + 1
# lo là λ nhỏ nhất thỏa điều kiện
```

Với `f` nguyên, mọi độ dốc là số nguyên nên ta chỉ cần xét `λ` nguyên.

In [7]:
# tìm số nguyên nhỏ nhất x sao cho x*x >= 200
lo, hi = 0, 200
while lo < hi:
    mid = (lo + hi) // 2
    if mid * mid >= 200:
        hi = mid
    else:
        lo = mid + 1
print(lo)             # 15

15


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1 (`nums = [4,1,-5,2]`, `m = 2`, `l = 1`, `r = 3`) bằng hàm `f`:

- `f(0) = 0`; `f(1) = 5` (đoạn `[4,1]`); `f(2) = 7` (`[4,1]` + `[2]`); `f(3) = 4 + 1 + 2 − ...`: ba đoạn dài ≥ 1 trong 4 phần tử, tốt nhất `[4],[1],[2]` = `7`; `f(4) = 4 + 1 − 5 + 2 = 2`.
- độ dốc: `d_1 = 5`, `d_2 = 2`, `d_3 = 0`, `d_4 = −5` (không tăng ⇒ lõm ✓).
- Đáp án = `max_{1 ≤ k ≤ 2} f(k) = 7`.

Aliens nhìn bài toán qua phạt `λ`:

- `λ = 0`: mỗi đoạn không bị phạt, tối ưu là `max f(k) = 7`, đạt tại `k = 2` hoặc `3` (hòa; ít đoạn nhất là `2`).
- `λ = 3`: chỉ những độ dốc `> 3` đáng giữ: `d_1 = 5` ⇒ chọn `1` đoạn, `g(3) = 5 − 3 = 2`.
- Ta muốn số đoạn đúng bằng `m = 2`: cần `λ` sao cho `d_2 ≥ λ` và `d_3 < λ`... tức `λ ∈ (0, 2]`.

Điểm quan trọng: không cần biết `f(k)` cho mọi `k`; chỉ cần **tìm `λ`** rồi suy ra `f(m)` từ `g(λ)`.

## 5. Cách 1 — Vét cạn: thử mọi tập đoạn

**Ý tưởng.** Đệ quy chọn các đoạn lần lượt, đoạn sau bắt đầu ở vị trí `≥` điểm kết thúc đoạn trước; mỗi lần đã chọn ít nhất một đoạn thì cập nhật đáp án. Dùng làm chân lý cho `n ≤ 9`.

In [8]:
def brute_force(nums, m, l, r):
    n = len(nums)
    P = [0]
    for x in nums:
        P.append(P[-1] + x)
    best = [None]

    def go(i, used, acc):
        if used >= 1 and (best[0] is None or acc > best[0]):
            best[0] = acc
        if used == m:
            return
        for start in range(i, n):
            for ln in range(l, r + 1):
                if start + ln <= n:
                    go(start + ln, used + 1, acc + P[start + ln] - P[start])

    go(0, 0, 0)
    return best[0]

for args, expected in examples:
    got = brute_force(**args)
    print(got, expected)
    assert got == expected

7 7
8 8
6 6
-1 -1


**Độ phức tạp.** Mũ theo `n`. Chỉ dùng để kiểm tra.

## 6. Bước đột phá

**Insight 1 — DP `O(n·m)` đã có sẵn nhưng quá chậm.** `f[t][i] = max(f[t][i−1], P[i] + max_{j∈[i−r,i−l]}(f[t−1][j] − P[j]))` (xem 7.1). Chiều `t` là vấn đề.

**Insight 2 — `f(k)` lõm theo `k`.** Độ dốc `f(k) − f(k−1)` không tăng. (Kiểm chứng thực nghiệm ở phần 3.3 và 12.)

**Insight 3 — phạt `λ` thay ràng buộc.** `g(λ) = max_k (f(k) − λk)` được tính bằng **một** DP `h[i]` không có chiều `t`:

`h[i] = max( h[i−1], P[i] + max_{j ∈ [i−r, i−l]} (h[j] − P[j]) − λ )`, `h[0] = 0`,

và `g(λ) = h[n]`. Bên cạnh `h[i]` ta giữ **số đoạn `c[i]`** của phương án đó.

**Insight 4 — phá hòa bằng "ít đoạn nhất".** Khi hai phương án cùng giá trị, chọn phương án **ít đoạn hơn**. Khi đó DP trả về `c_min(λ) = #{k : d_k > λ}`: không tăng theo `λ`. Cách làm gọn: so sánh cặp `(giá trị, −số đoạn)` theo thứ tự từ điển, hoặc gộp thành một số nguyên `giá_trị · M − số_đoạn` với `M > n`.

**Insight 5 — tìm `λ` và khôi phục `f(m)`.** Tìm `λ` nguyên **nhỏ nhất ≥ 0** sao cho `c_min(λ) ≤ m` (nhị phân vì `c_min` không tăng theo `λ`). Khi đó:

- nếu `λ = 0`: trả `g(0)` (số đoạn tối ưu đã `≤ m` nên `max_k f(k)` đạt trong giới hạn `m`);
- nếu `λ > 0`: `c_min(λ−1) > m` nên `m` nằm giữa `c_min(λ)` và `c_max(λ)` (mọi số đoạn trong khoảng đó cùng tối ưu cho `λ`). Vậy `f(m) − λ·m = g(λ)` ⇒ **`f(m) = g(λ) + λ·m`**, và vì `λ > 0` nên `f` đang tăng đến `m`, do đó `max_{k ≤ m} f(k) = f(m)`.

**Insight 6 — "ít nhất một đoạn".** Tính riêng `f(1)` (đoạn đơn tốt nhất). Nếu `f(1) ≤ 0` (hoặc `m = 1`), đáp án là `f(1)` (vì `f` lõm và `f(0) = 0`, mọi `f(k)` với `k ≥ 1` không vượt `f(1)` khi `d_1 ≤ 0`). Nếu `f(1) > 0` thì `max_{1≤k≤m} f(k) = max_{0≤k≤m} f(k)`, tức dùng Aliens như trên.

**Insight 7 — cận của nhị phân.** `d_1 = f(1) ≤ 10^{10}` là độ dốc lớn nhất; với `λ = f(1) + 1`, không đoạn nào đáng chọn (`c_min = 0 ≤ m`). Nên tìm `λ ∈ [0, f(1) + 1]`, khoảng `34` lần lặp.

**Quy tắc "nếu… thì…":**

- Nếu `m > K = n // l` thì `c_min(0) ≤ K < m` ⇒ `λ = 0` ⇒ trả `g(0)`.
- Nếu `c_min(λ) ≤ m` thì thu hẹp `hi = λ`; ngược lại `lo = λ + 1`.
- Giá trị `g` có thể rất lớn (`~10^{10}`) nhưng nằm gọn trong số nguyên Python.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP `O(n·m)` với hàng đợi đơn điệu (nền tảng, đúng nhưng chậm)

**Giải thích từng đoạn.** `prev = f[t−1]`, `cur = f[t]`. Với mỗi `i`, đưa `j = i − l` vào deque (khóa `prev[j] − P[j]`), bỏ `j < i − r` ở đầu, rồi `cur[i] = max(cur[i−1], P[i] + max_deque)`. Đáp án `max_{t ≥ 1} cur[n]`. Đây là lời giải của bài I.

In [9]:
def dp_deque(nums, m, l, r):
    n = len(nums)
    P = [0] * (n + 1)
    for i, x in enumerate(nums):
        P[i + 1] = P[i] + x
    prev = [0] * (n + 1)
    best = NEG
    for t in range(1, m + 1):
        cur = [NEG] * (n + 1)
        dq = deque()
        for i in range(1, n + 1):
            j = i - l
            if j >= 0 and prev[j] > NEG:
                v = prev[j] - P[j]
                while dq and dq[-1][0] <= v:
                    dq.pop()
                dq.append((v, j))
            while dq and dq[0][1] < i - r:
                dq.popleft()
            cur[i] = cur[i - 1]
            if dq and P[i] + dq[0][0] > cur[i]:
                cur[i] = P[i] + dq[0][0]
        if cur[n] > best:
            best = cur[n]
        prev = cur
    return best

for args, expected in examples:
    got = dp_deque(**args)
    print(got, expected)
    assert got == expected

7 7
8 8
6 6
-1 -1


**Độ phức tạp.** `O(n · m)` thời gian, `O(n)` bộ nhớ. Với `n = 10^5`, `m = 10^5`: `10^{10}` bước — không thể. Ta đo thử để thấy tốc độ tăng.

In [10]:
import time
rng = random.Random(1)
elapsed = None
for n, m in [(500, 500), (1000, 1000), (2000, 2000)]:
    nums = [rng.randint(-10**5, 10**5) for _ in range(n)]
    t0 = time.time()
    dp_deque(nums, m, 1, 3)
    elapsed = (time.time() - t0, n * m)
    print(f"n=m={n}: {elapsed[0]:.2f}s  (n*m = {n*m:.1e})")
est = elapsed[0] / elapsed[1] * 10**10
print(f"ngoại suy n=m=1e5 (n*m = 1e10): khoảng {est/60:.0f} phút — vượt xa giới hạn vài giây của LeetCode")

n=m=500: 0.05s  (n*m = 2.5e+05)


n=m=1000: 0.20s  (n*m = 1.0e+06)


n=m=2000: 0.83s  (n*m = 4.0e+06)
ngoại suy n=m=1e5 (n*m = 1e10): khoảng 35 phút — vượt xa giới hạn vài giây của LeetCode


### 7.2. Cách 3 — Aliens trick, bản dễ hiểu: DP với cặp `(giá trị, −số đoạn)`

**Giải thích từng đoạn.**

1. `penalty_pass(λ)`: DP một chiều `h[i]` = cặp `(giá trị tốt nhất, −số đoạn)` cho `i` phần tử đầu, so sánh **theo thứ tự từ điển** (giá trị lớn hơn thắng; hòa thì `−số đoạn` lớn hơn tức số đoạn **ít** hơn thắng). `h[0] = (0, 0)`.
2. Chuyển trạng thái: `h[i] = max(h[i−1], (g + P[i] − λ, c − 1))` với `(g, c)` là max của `(h[j].giá_trị − P[j], h[j].−số_đoạn)` trên cửa sổ `j ∈ [i−r, i−l]` (thêm một đoạn: giá trị `+ P[i] − P[j] − λ`, số đoạn `+1` tức `−số_đoạn − 1`).
3. Nhị phân `λ ∈ [0, f(1) + 1]` tìm `λ` nhỏ nhất có `số đoạn ≤ m`.
4. Khôi phục: `λ = 0` ⇒ `g(0)`; `λ > 0` ⇒ `g(λ) + λ·m`.
5. Trước đó xử lý `f(1)` (đoạn đơn tốt nhất) bằng cửa sổ trượt min trên `P`.

In [11]:
def best_single(P, n, l, r):
    """f(1): tổng lớn nhất của một đoạn dài [l, r], bằng sliding min của P[j]."""
    best = NEG
    dq = deque()                                  # (P[j], j), giá trị tăng dần
    for i in range(l, n + 1):
        j = i - l
        while dq and dq[-1][0] >= P[j]:
            dq.pop()
        dq.append((P[j], j))
        while dq[0][1] < i - r:
            dq.popleft()
        best = max(best, P[i] - dq[0][0])
    return best

def penalty_pass_tuple(P, n, l, r, lam):
    """Trả về (g(λ), số đoạn nhỏ nhất đạt g(λ)). h[i] = (giá trị, -số đoạn) so sánh từ điển."""
    h = [(0, 0)] * (n + 1)
    dq = deque()                                  # ((giá trị, -số đoạn), j) giảm dần
    for i in range(1, n + 1):
        j = i - l
        if j >= 0:
            key = (h[j][0] - P[j], h[j][1])
            while dq and dq[-1][0] <= key:
                dq.pop()
            dq.append((key, j))
        while dq and dq[0][1] < i - r:
            dq.popleft()
        best = h[i - 1]
        if dq:
            v, c = dq[0][0]
            cand = (v + P[i] - lam, c - 1)
            if cand > best:
                best = cand
        h[i] = best
    return h[n][0], -h[n][1]

def aliens_tuple(nums, m, l, r):
    n = len(nums)
    P = [0] * (n + 1)
    for i, x in enumerate(nums):
        P[i + 1] = P[i] + x
    f1 = best_single(P, n, l, r)
    if m == 1 or f1 <= 0:
        return f1
    lo, hi = 0, f1 + 1                            # tại hi: không đoạn nào đáng chọn
    while lo < hi:
        mid = (lo + hi) // 2
        if penalty_pass_tuple(P, n, l, r, mid)[1] <= m:
            hi = mid
        else:
            lo = mid + 1
    g, _ = penalty_pass_tuple(P, n, l, r, lo)
    return g if lo == 0 else g + lo * m

for args, expected in examples:
    got = aliens_tuple(**args)
    print(got, expected)
    assert got == expected

7 7
8 8
6 6
-1 -1


**Độ phức tạp.** Mỗi lượt DP `O(n)`; nhị phân `≈ 34` lượt (vì `λ ≤ 10^{10}`) ⇒ `O(n · log V)`, `V = f(1) + 1 ≤ 10^{10} + 1`. Hằng số của tuple lớn hơn số nguyên, nên ta tối ưu ở cách 4.

### 7.3. Cách 4 — Aliens tối ưu: mã hóa khóa số nguyên `giá_trị · M − số_đoạn` (bản chính)

**Ý tưởng.** Thay cặp `(giá trị, −số đoạn)` bằng **một số nguyên** `key = giá_trị · M − số_đoạn` với `M = n + 2`. Vì `0 ≤ số_đoạn ≤ n < M`, so sánh `key` tương đương so sánh từ điển: ưu tiên giá trị, hòa thì số đoạn ít hơn.

- Thêm một đoạn: `key' = key_j − P[j]·M + P[i]·M − λ·M − 1` (giá trị `+ P[i] − P[j] − λ`, số đoạn `+1`).
- Giải mã cuối: `c = (−key) mod M`, `giá_trị = (key + c) / M`.

Nhờ đó deque lưu số nguyên thay vì tuple: nhanh hơn rõ rệt.

In [12]:
def penalty_pass_key(P, n, l, r, lam, M):
    h = [0] * (n + 1)                              # h[i] = giá_trị * M - số_đoạn
    dq = deque()                                   # (h[j] - P[j]*M, j), giảm dần
    lamM = lam * M
    for i in range(1, n + 1):
        j = i - l
        if j >= 0:
            v = h[j] - P[j] * M
            while dq and dq[-1][0] <= v:
                dq.pop()
            dq.append((v, j))
        while dq and dq[0][1] < i - r:
            dq.popleft()
        best = h[i - 1]
        if dq:
            cand = dq[0][0] + P[i] * M - lamM - 1  # +1 đoạn: key giảm 1
            if cand > best:
                best = cand
        h[i] = best
    key = h[n]
    c = (-key) % M
    return (key + c) // M, c

def aliens_key(nums, m, l, r):
    n = len(nums)
    P = [0] * (n + 1)
    for i, x in enumerate(nums):
        P[i + 1] = P[i] + x
    f1 = best_single(P, n, l, r)
    if m == 1 or f1 <= 0:
        return f1
    M_ = n + 2
    lo, hi = 0, f1 + 1
    while lo < hi:
        mid = (lo + hi) // 2
        if penalty_pass_key(P, n, l, r, mid, M_)[1] <= m:
            hi = mid
        else:
            lo = mid + 1
    g, _ = penalty_pass_key(P, n, l, r, lo, M_)
    return g if lo == 0 else g + lo * m

for args, expected in examples:
    got = aliens_key(**args)
    print(got, expected)
    assert got == expected

7 7
8 8
6 6
-1 -1


**Độ phức tạp.** `O(n · log V)` thời gian, `O(n)` bộ nhớ. Với `n = 10^5`: khoảng `34 × 10^5 ≈ 3.4 · 10^6` bước deque — khoảng 1–3 giây trong Python.

## 8. TRACE từng bước

Ta trace `aliens_key` cho ví dụ 1 (`nums = [4,1,-5,2]`, `m = 2`, `l = 1`, `r = 3`). In `f(1)`, từng bước nhị phân `λ` (với `g(λ)` và số đoạn `c(λ)`), và bước khôi phục.

Bảng mong đợi (độ dốc `d = [5, 2, 0, −5]`, `f(1) = 5`):

| `λ` thử | `g(λ)` | `c_min(λ)` = số độ dốc `> λ` | so với `m = 2` |
|---|---|---|---|
| 3 (`lo=0, hi=6`) | `2` (`5 − 3`) | `1` (chỉ `5`) | `≤ 2` ⇒ `hi = 3` |
| 1 (`lo=0, hi=3`) | `5` (`7 − 2·1`) | `2` (`5, 2`) | `≤ 2` ⇒ `hi = 1` |
| 0 (`lo=0, hi=1`) | `7` | `2` (`5, 2`; độ dốc `0` không `> 0`) | `≤ 2` ⇒ `hi = 0` |

Kết thúc `λ = 0` ⇒ trả `g(0) = 7` ✓.

In [13]:
def trace_aliens(nums, m, l, r):
    n = len(nums)
    P = [0] * (n + 1)
    for i, x in enumerate(nums):
        P[i + 1] = P[i] + x
    f1 = best_single(P, n, l, r)
    print(f"P = {P}, f(1) = {f1}")
    if m == 1 or f1 <= 0:
        print("trả về f(1) ngay:", f1)
        return f1
    M_ = n + 2
    lo, hi = 0, f1 + 1
    step = 0
    while lo < hi:
        mid = (lo + hi) // 2
        g, c = penalty_pass_key(P, n, l, r, mid, M_)
        step += 1
        verdict = f"c={c} <= m={m} -> hi = {mid}" if c <= m else f"c={c} > m={m} -> lo = {mid + 1}"
        print(f"bước {step}: [lo={lo}, hi={hi}) thử λ={mid}: g(λ)={g}, {verdict}")
        if c <= m:
            hi = mid
        else:
            lo = mid + 1
    g, c = penalty_pass_key(P, n, l, r, lo, M_)
    ans = g if lo == 0 else g + lo * m
    print(f"λ* = {lo}, g(λ*) = {g}, c(λ*) = {c}; đáp án = " + ("g(0)" if lo == 0 else f"g + λ*·m = {g} + {lo}*{m}") + f" = {ans}")
    return ans

trace_aliens([4, 1, -5, 2], 2, 1, 3)

P = [0, 4, 5, 0, 2], f(1) = 5
bước 1: [lo=0, hi=6) thử λ=3: g(λ)=2, c=1 <= m=2 -> hi = 3
bước 2: [lo=0, hi=3) thử λ=1: g(λ)=5, c=2 <= m=2 -> hi = 1
bước 3: [lo=0, hi=1) thử λ=0: g(λ)=7, c=2 <= m=2 -> hi = 0
λ* = 0, g(λ*) = 7, c(λ*) = 2; đáp án = g(0) = 7


7

In [14]:
# Trace khi m nhỏ hơn số đoạn tối ưu: m = 1 ... và một ví dụ m=2 buộc λ > 0
trace_aliens([4, 1, -5, 2, 6, -3, 3, 1], 2, 1, 3)
print("vét cạn:", dp_deque([4, 1, -5, 2, 6, -3, 3, 1], 2, 1, 3))
print("độ dốc f:", [exact_values([4, 1, -5, 2, 6, -3, 3, 1], 1, 3)[k] - exact_values([4, 1, -5, 2, 6, -3, 3, 1], 1, 3)[k - 1] for k in range(1, 9)])

P = [0, 4, 5, 0, 2, 8, 5, 8, 9], f(1) = 8
bước 1: [lo=0, hi=9) thử λ=4: g(λ)=5, c=2 <= m=2 -> hi = 4
bước 2: [lo=0, hi=4) thử λ=2: g(λ)=11, c=3 > m=2 -> lo = 3
bước 3: [lo=3, hi=4) thử λ=3: g(λ)=8, c=3 > m=2 -> lo = 4
λ* = 4, g(λ*) = 5, c(λ*) = 2; đáp án = g + λ*·m = 5 + 4*2 = 13
vét cạn: 13
độ dốc f: [8, 5, 4, 0, 0, 0, -3, -5]


Ở ví dụ thứ hai, độ dốc của `f` là các số dương lớn dần rồi giảm; với `m = 2` nhị phân dừng ở `λ > 0` và đáp án được khôi phục bằng `g(λ) + λ·m` (không còn là `g(0)`), khớp `dp_deque`.

## 9. Code nộp LeetCode

Dùng cách 4. Viết gọn trong một phương thức với hai hàm phụ: `best_single` (đoạn đơn tốt nhất) và `solve(λ)` (một lượt DP phạt).

In [15]:
from collections import deque

class Solution:
    def maximumSum(self, nums: list[int], m: int, l: int, r: int) -> int:
        n = len(nums)
        P = [0] * (n + 1)
        for i, x in enumerate(nums):
            P[i + 1] = P[i] + x

        # f(1): đoạn đơn tốt nhất = max (P[i] - min P[j]) với j trong [i-r, i-l]
        f1 = None
        dq = deque()
        for i in range(l, n + 1):
            j = i - l
            while dq and dq[-1][0] >= P[j]:
                dq.pop()
            dq.append((P[j], j))
            while dq[0][1] < i - r:
                dq.popleft()
            cand = P[i] - dq[0][0]
            if f1 is None or cand > f1:
                f1 = cand
        if m == 1 or f1 <= 0:                       # f lõm: nếu độ dốc đầu <= 0 thì f(1) là tốt nhất
            return f1

        M = n + 2                                    # M > số đoạn tối đa

        def solve(lam):
            """(g(lam), số đoạn nhỏ nhất đạt g(lam)); key = giá_trị * M - số_đoạn."""
            h = [0] * (n + 1)
            dq = deque()
            lamM = lam * M
            for i in range(1, n + 1):
                j = i - l
                if j >= 0:
                    v = h[j] - P[j] * M
                    while dq and dq[-1][0] <= v:
                        dq.pop()
                    dq.append((v, j))
                while dq and dq[0][1] < i - r:
                    dq.popleft()
                best = h[i - 1]
                if dq:
                    cand = dq[0][0] + P[i] * M - lamM - 1
                    if cand > best:
                        best = cand
                h[i] = best
            key = h[n]
            c = (-key) % M
            return (key + c) // M, c

        lo, hi = 0, f1 + 1                           # λ nhỏ nhất có số đoạn <= m
        while lo < hi:
            mid = (lo + hi) // 2
            if solve(mid)[1] <= m:
                hi = mid
            else:
                lo = mid + 1
        g, _ = solve(lo)
        return g if lo == 0 else g + lo * m


sol = Solution()
for args, expected in examples:
    got = sol.maximumSum(**args)
    print(got, expected)
    assert got == expected

7 7
8 8
6 6
-1 -1


## 10. Vì sao thuật toán đúng

**Tính lõm là điều kiện then chốt.** Giả sử `f(0) = 0, f(1), ..., f(K)` lõm (độ dốc `d_k` không tăng). (Ta đã kiểm chứng thực nghiệm; chứng minh chặt cần thêm lập luận trao đổi hoặc mô hình luồng và không nằm trong phạm vi notebook này. Nếu `f` không lõm, Aliens có thể cho đáp án sai — vì vậy phần 12 đối chiếu với DP `O(n·m)` trên rất nhiều input.)

**Phạt `λ` và số đoạn tối ưu.** `φ(k) = f(k) − λk` có độ dốc `d_k − λ`; vì `d_k` không tăng nên `φ` tăng khi `d_k > λ`, đứng yên khi `d_k = λ`, giảm khi `d_k < λ`. Do đó tập nghiệm tối ưu `argmax φ` là một đoạn `[c_min(λ), c_max(λ)]` với `c_min(λ) = #{k : d_k > λ}` và `c_max(λ) = #{k : d_k ≥ λ}`. DP phạt tối ưu `g(λ) = max_k φ(k)` (kể cả `k = 0`), và nhờ phá hòa "ít đoạn nhất" nó trả `c_min(λ)`.

**DP phạt đúng.** `h[i]` là giá trị tốt nhất (đã trừ `λ` mỗi đoạn) cho `i` phần tử đầu với **số đoạn tùy ý**: hoặc `i−1` không thuộc đoạn nào (`h[i−1]`), hoặc đoạn cuối là `[j, i)` với `i − j ∈ [l, r]` (`h[j] + P[i] − P[j] − λ`). Khóa số nguyên `key = giá_trị · M − số_đoạn` có thứ tự trùng thứ tự từ điển `(giá trị, −số đoạn)` vì `số_đoạn < M`; phép cộng "thêm một đoạn" trừ `λ·M + 1` đúng ý.

**Nhị phân.** `c_min(λ)` không tăng theo `λ`; ta tìm `λ*` nguyên nhỏ nhất `≥ 0` có `c_min(λ*) ≤ m` (tại `λ = f(1) + 1` thì `c_min = 0 ≤ m`, đảm bảo tồn tại). Độ dốc nguyên nên mọi `λ` nguyên là đủ.

**Khôi phục `f(m)`.** 

- `λ* = 0`: `g(0) = max_k f(k)` đạt tại `c_min(0) ≤ m` đoạn ⇒ `max_{k ≤ m} f(k) = g(0)`.
- `λ* > 0`: `c_min(λ* − 1) > m` ⇒ `#{d_k ≥ λ*} > m` ⇒ `c_max(λ*) > m ≥ c_min(λ*)` ⇒ `m ∈ [c_min(λ*), c_max(λ*)]` ⇒ `φ(m) = g(λ*)` ⇒ `f(m) = g(λ*) + λ*·m`. Vì `d_m ≥ λ* > 0` nên `f` tăng đến `m`, nên `max_{k ≤ m} f(k) = f(m)`.

**Ít nhất một đoạn.** Nếu `d_1 = f(1) > 0` thì `f(1) > f(0)`, nên `max_{0≤k≤m} f(k) = max_{1≤k≤m} f(k)`: Aliens (bao gồm `k = 0`) trả đúng đáp án "ít nhất một". Nếu `f(1) ≤ 0` thì `d_1 ≤ 0` và theo tính lõm `f(k) ≤ f(1)` với mọi `k ≥ 1`; đáp án là `f(1)`.

**`m` lớn hơn `K`.** Khi `m > K = n // l` thì `c_min(λ) ≤ K < m` với mọi `λ ≥ 0` nên `λ* = 0` và trả `g(0)` — đúng vì không thể chọn hơn `K` đoạn.

## 11. Phân tích Time/Space

Đặt `n = len(nums)`, `V = f(1) + 1 ≤ 10^{10} + 1`.

| Cách | Time | Space |
|---|---|---|
| Vét cạn | mũ | `O(n)` |
| DP `O(n·m)` + deque | `O(n·m)` | `O(n)` |
| Aliens (tuple) | `O(n · log V)` (hằng số lớn) | `O(n)` |
| Aliens (khóa số nguyên, nộp) | `O(n · log V)` | `O(n)` |

`log2(10^{10}) ≈ 34` lượt DP, mỗi lượt `O(n)` ⇒ `~3.4 · 10^6` thao tác deque cho `n = 10^5`.

In [16]:
import time, random
rng = random.Random(7)
n = 100000
nums = [rng.randint(-10**5, 10**5) for _ in range(n)]
for (m, l, r) in [(50, 5, 100), (50000, 1, 3), (1000, 10, 2000), (100000, 1, 100000), (3, 100, 100)]:
    t0 = time.time()
    res = Solution().maximumSum(nums, m, l, r)
    print(f"m={m:>6}, l={l:>3}, r={r:>6}: {res}  ({time.time()-t0:.2f}s)")
# toàn số dương: nên phủ càng nhiều càng tốt; toàn âm: đáp án là số âm lớn nhất trong cửa sổ độ dài l..r
pos = [rng.randint(1, 10**5) for _ in range(n)]
print("toàn dương m=n, l=1:", Solution().maximumSum(pos, n, 1, 1), "= sum:", sum(pos))
neg = [rng.randint(-10**5, -1) for _ in range(n)]
print("toàn âm   m=n, l=1:", Solution().maximumSum(neg, n, 1, 1), "= max:", max(neg))

m=    50, l=  5, r=   100: 81658151  (0.94s)


m= 50000, l=  1, r=     3: 2490359197  (0.87s)


m=  1000, l= 10, r=  2000: 742381668  (1.09s)


m=100000, l=  1, r=100000: 2490359197  (1.13s)


m=     3, l=100, r=   100: 5936003  (0.92s)


toàn dương m=n, l=1: 4987019326 = sum: 4987019326
toàn âm   m=n, l=1: -2 = max: -2


## 12. Bộ test

Chiến lược test:

1. 4 ví dụ của đề (assert).
2. Biên: `n = 1`; `m = 1`; `l = r`; `l = r = n`; toàn âm; toàn dương; toàn `0`; `m > n // l`.
3. Phản ví dụ: không phá hòa (quên "ít đoạn nhất"), hoặc quên xử lý `f(1) ≤ 0` (phần 13).
4. Đối chiếu vét cạn, DP `O(n·m)`, Aliens tuple, Aliens khóa và `Solution` trên input ngẫu nhiên có seed cố định — đây cũng là cách **kiểm chứng tính lõm** (nếu `f` không lõm Aliens sẽ lệch DP).

In [17]:
sol = Solution()
for args, expected in examples:
    assert sol.maximumSum(**args) == expected

assert sol.maximumSum([5], 1, 1, 1) == 5
assert sol.maximumSum([-5], 1, 1, 1) == -5
assert sol.maximumSum([1, 2, 3, 4], 4, 1, 1) == 10
assert sol.maximumSum([1, 2, 3, 4], 1, 4, 4) == 10
assert sol.maximumSum([1, 2, 3, 4], 2, 2, 2) == 10
assert sol.maximumSum([1, -2, 3, -4, 5], 2, 1, 1) == 8
assert sol.maximumSum([-1, -2, -3], 3, 1, 1) == -1
assert sol.maximumSum([0, 0, 0, 0], 2, 1, 4) == 0
assert sol.maximumSum([2, 2, 2, 2, 2], 5, 2, 2) == 8              # m = 5 > n//l = 2: tối đa 2 đoạn dài 2
print("test cố định OK")

test cố định OK


In [18]:
import random
rng = random.Random(99)
cnt = 0
for _ in range(2000):
    n = rng.randint(1, 8)
    nums = [rng.randint(-9, 9) for _ in range(n)]
    l = rng.randint(1, n); r = rng.randint(l, n); m = rng.randint(1, n)
    expected = brute_force(nums, m, l, r)
    assert dp_deque(nums, m, l, r) == expected, (nums, m, l, r)
    assert aliens_tuple(nums, m, l, r) == expected, (nums, m, l, r)
    assert aliens_key(nums, m, l, r) == expected, (nums, m, l, r)
    assert Solution().maximumSum(nums, m, l, r) == expected, (nums, m, l, r)
    cnt += 1
print("đã đối chiếu", cnt, "input ngẫu nhiên nhỏ: tất cả khớp vét cạn")

đã đối chiếu 2000 input ngẫu nhiên nhỏ: tất cả khớp vét cạn


In [19]:
# Input vừa (n tới 60, giá trị lớn/nhỏ): so với DP O(n*m) — kiểm chứng tính lõm trong thực tế
rng = random.Random(1234)
cnt = 0
for _ in range(1500):
    n = rng.randint(1, 60)
    V = rng.choice([3, 20, 1000, 10**5])
    nums = [rng.randint(-V, V) for _ in range(n)]
    l = rng.randint(1, min(n, rng.choice([1, 2, 4, 8, n])))
    r = rng.randint(l, min(n, l + rng.choice([0, 1, 3, 10, n])))
    m = rng.randint(1, n)
    e = dp_deque(nums, m, l, r)
    assert Solution().maximumSum(nums, m, l, r) == e == aliens_tuple(nums, m, l, r), (nums, m, l, r)
    cnt += 1
print("Aliens == DP O(n·m) trên", cnt, "input vừa")

Aliens == DP O(n·m) trên 1500 input vừa


In [20]:
# Kiểm tra lõm trực tiếp trên nhiều input: độ dốc f(k)-f(k-1) không tăng
rng = random.Random(5)
bad = 0
for _ in range(3000):
    n = rng.randint(1, 14)
    l = rng.randint(1, min(n, 5)); r = rng.randint(l, n)
    nums = [rng.randint(-8, 8) for _ in range(n)]
    v = exact_values(nums, l, r)
    d = [v[k] - v[k - 1] for k in range(1, len(v))]
    bad += any(d[k] < d[k + 1] for k in range(len(d) - 1))
print("input vi phạm tính lõm:", bad, "/ 3000")
assert bad == 0

input vi phạm tính lõm: 0 / 3000


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Không phá hòa bằng "ít đoạn nhất" (hoặc phá hòa không nhất quán)

**Code sai:** trả về số đoạn tùy ý khi có nhiều phương án cùng giá trị, rồi so sánh với `m`.
**Vì sao sai:** với `λ` bằng một độ dốc có nhiều số đoạn cùng tối ưu; nếu số đoạn trả về lúc `≤ m` lúc `> m` không nhất quán thì `c(λ)` không đơn điệu, nhị phân hỏng. Quy tắc cố định "ít đoạn nhất" làm `c_min(λ)` không tăng theo `λ`.
**Code đúng:** khóa `giá_trị · M − số_đoạn`.

In [21]:
def wrong_no_tiebreak(nums, m, l, r):
    """Aliens nhưng khóa chỉ là giá trị (không theo dõi số đoạn nhất quán): đếm số đoạn của một phương án BẤT KỲ."""
    n = len(nums)
    P = [0] * (n + 1)
    for i, x in enumerate(nums):
        P[i + 1] = P[i] + x
    f1 = best_single(P, n, l, r)
    if m == 1 or f1 <= 0:
        return f1

    def run(lam):
        h = [(0, 0)] * (n + 1)                     # (giá trị, số đoạn) ; hòa thì lấy phương án MỚI (nhiều đoạn hơn)
        dq = deque()
        for i in range(1, n + 1):
            j = i - l
            if j >= 0:
                key = h[j][0] - P[j]
                while dq and dq[-1][0] <= key:
                    dq.pop()
                dq.append((key, h[j][1], j))
            while dq and dq[0][2] < i - r:
                dq.popleft()
            best = h[i - 1]
            if dq:
                cand = (dq[0][0] + P[i] - lam, dq[0][1] + 1)
                if cand[0] >= best[0]:                # SAI: hòa thì ưu tiên nhiều đoạn hơn
                    best = cand
            h[i] = best
        return h[n]

    lo, hi = 0, f1 + 1
    while lo < hi:
        mid = (lo + hi) // 2
        if run(mid)[1] <= m:
            hi = mid
        else:
            lo = mid + 1
    g, _ = run(lo)
    return g if lo == 0 else g + lo * m

rng = random.Random(3)
shown = 0
for _ in range(3000):
    n = rng.randint(2, 9)
    nums = [rng.randint(-4, 6) for _ in range(n)]
    l = rng.randint(1, n); r = rng.randint(l, n); m = rng.randint(1, n)
    w, c = wrong_no_tiebreak(nums, m, l, r), Solution().maximumSum(nums, m, l, r)
    if w != c and shown < 3:
        print("phản ví dụ:", nums, (m, l, r), "sai:", w, "đúng:", c)
        shown += 1
print("đã tìm", shown, "phản ví dụ" if shown else "(không thấy phản ví dụ trong mẫu này)")

phản ví dụ: [2, 0, 4, 0, 4, 1] (5, 1, 4) sai: 14 đúng: 11
phản ví dụ: [1, 5, 2, 1, 4, 2] (5, 1, 4) sai: 18 đúng: 15
phản ví dụ: [-3, 1, 6, 5, 4, 4] (4, 1, 6) sai: 23 đúng: 20
đã tìm 3 phản ví dụ


### Lỗi 2 — Trả về `g(λ)` thay vì `g(λ) + λ·m`

**Code sai:** `return g` sau nhị phân.
**Vì sao sai:** `g(λ)` đã **trừ** `λ` cho mỗi đoạn dùng; muốn tổng thật của đúng `m` đoạn phải **cộng lại** `λ · m`. Chỉ khi `λ = 0` thì hai giá trị trùng.
**Code đúng:** `g + lo * m` (khi `lo > 0`).

In [22]:
def wrong_forget_add_back(nums, m, l, r):
    n = len(nums)
    P = [0] * (n + 1)
    for i, x in enumerate(nums):
        P[i + 1] = P[i] + x
    f1 = best_single(P, n, l, r)
    if m == 1 or f1 <= 0:
        return f1
    M_ = n + 2
    lo, hi = 0, f1 + 1
    while lo < hi:
        mid = (lo + hi) // 2
        if penalty_pass_key(P, n, l, r, mid, M_)[1] <= m:
            hi = mid
        else:
            lo = mid + 1
    return penalty_pass_key(P, n, l, r, lo, M_)[0]      # SAI: quên + lo * m

nums_demo = [4, 1, -5, 2, 6, -3, 3, 1]
print("sai :", wrong_forget_add_back(nums_demo, 2, 1, 3), " đúng:", Solution().maximumSum(nums_demo, 2, 1, 3), " DP:", dp_deque(nums_demo, 2, 1, 3))

sai : 5  đúng: 13  DP: 13


### Lỗi 3 — Quên xử lý trường hợp `f(1) ≤ 0` (toàn âm)

**Code sai:** chạy Aliens với `hi = f(1) + 1 ≤ 1` hoặc trả `g(0) = 0`.
**Vì sao sai:** `g(0)` tính cả `k = 0` (không chọn đoạn nào) nên trả `0`, trong khi đề bắt buộc chọn ít nhất một đoạn nên đáp án âm.
**Code đúng:** nếu `f(1) ≤ 0` trả `f(1)`.

In [23]:
def wrong_all_negative(nums, m, l, r):
    n = len(nums)
    P = [0] * (n + 1)
    for i, x in enumerate(nums):
        P[i + 1] = P[i] + x
    M_ = n + 2
    return penalty_pass_key(P, n, l, r, 0, M_)[0]        # SAI: g(0) gồm cả phương án rỗng

print("sai :", wrong_all_negative(**examples[3][0]), " đúng:", Solution().maximumSum(**examples[3][0]))

sai : 0  đúng: -1


### Lỗi 4 — Cận nhị phân sai (`hi` quá nhỏ)

**Code sai:** `hi = 0` hoặc `hi = max(nums)`.
**Vì sao sai:** độ dốc lớn nhất là `f(1)` — tổng một đoạn, có thể lên `r · max(nums)`. Nếu `hi` nhỏ hơn thì có thể không tồn tại `λ` thỏa `c ≤ m` trong khoảng, nhị phân dừng ở biên sai.
**Code đúng:** `hi = f(1) + 1`.

### Lỗi 5 — `M` (hệ số mã hóa) không đủ lớn

**Code sai:** `M = 10` hoặc `M = m`.
**Vì sao sai:** số đoạn có thể tới `n`; nếu `M ≤ số đoạn` thì giải mã `c = (−key) mod M` sai và so sánh khóa không còn đúng thứ tự từ điển.
**Code đúng:** `M = n + 2`.

## 14. Mẹo & nhận dạng pattern

- **"Chọn đúng/tối đa `k` đối tượng để tối đa tổng, `k` lớn"** + bạn nghi `f(k)` **lõm** → thử **Aliens trick**: thêm phạt `λ` mỗi đối tượng, nhị phân `λ`, mỗi lần một DP không chiều `k`.
- Các bài cùng họ: *Best Time to Buy and Sell Stock IV* (`k` lớn), *chọn `k` đoạn không giao nhau tổng lớn nhất*, bài *Aliens* (IOI 2016), *Minimum cost to split in k groups* có tính lồi.
- Với tối đa hóa và lõm: `f(m) = g(λ) + λ·m`; với tối thiểu hóa và lồi: `f(m) = g(λ) + λ·m` tương tự (đổi chiều so sánh).
- **Phá hòa nhất quán** (ít đoạn nhất hoặc nhiều đoạn nhất) rồi chọn công thức tìm `λ` tương ứng; trộn lẫn hai quy tắc là lỗi hay gặp.
- Khi giá trị nguyên thì chỉ cần nhị phân `λ` nguyên; nếu giá trị thực cần nhị phân thực và xử lý sai số.
- **Kiểm chứng tính lõm bằng DP `O(n·m)` trên input nhỏ ngẫu nhiên** trước khi tin vào Aliens: đây là bước an toàn nhất.
- Dùng khóa số nguyên `giá_trị · M − số_đoạn` để gộp hai tiêu chí vào một phép so sánh nhanh.

## 15. Biến thể

1. **Đúng `m` đoạn (không phải tối đa):** tìm `λ` có thể âm, `f(m) = g(λ) + λ·m`; cần `m ≤ n // l` (Bài 5).
2. **Aliens với "nhiều đoạn nhất" khi hòa:** tìm `λ` lớn nhất có `c_max(λ) ≥ m`; công thức khôi phục giống nhau.
3. **Giới hạn độ dài khác (`l = 1`, `r = n`):** bài *k đoạn không giao nhau tổng lớn nhất* kinh điển; DP phạt đơn giản hơn (không cần deque).
4. **Tối thiểu hóa tổng** (đoạn có tổng nhỏ nhất): đổi dấu `nums`.
5. **Có thêm điều kiện khoảng cách tối thiểu giữa các đoạn:** DP phạt dùng `h[j − g]` thay `h[j]`; `f` vẫn lõm trong nhiều biến thể nhưng cần kiểm chứng.

## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | thử mọi tập đoạn | mũ | `O(n)` | đối chiếu `n ≤ 9` |
| 2. DP `O(n·m)` + deque | `f[t][i]` với cửa sổ trượt | `O(n·m)` | `O(n)` | `n·m ≲ 10^7`, kiểm chứng Aliens |
| 3. Aliens (tuple) | phạt `λ`, nhị phân, `(giá trị, −số đoạn)` | `O(n log V)` | `O(n)` | hiểu thuật toán |
| 4. Aliens (khóa số nguyên) | như 3 nhưng khóa `giá_trị·M − số_đoạn` | `O(n log V)` | `O(n)` | **nộp bài** |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** Với `f(k) = [0, 5, 7, 7, 2]` (cho `k = 0..4`), độ dốc `d_k` là gì? `f` có lõm không?

**Bài 2 (tính tay).** Với `f(k)` ở Bài 1 và `λ = 3`, tính `g(λ) = max_k (f(k) − 3k)` và số đoạn tối ưu nhỏ nhất.

**Bài 3 (điền TODO).** Hoàn thiện `decode(key, M)` trả `(giá_trị, số_đoạn)` từ `key = giá_trị · M − số_đoạn`.

**Bài 4 (tự viết lại).** Viết `my_best_single(nums, l, r)` trả tổng lớn nhất của **một** đoạn dài `[l, r]` (có thể dùng vét cạn `O(n·r)`).

**Bài 5 (biến thể).** Viết `exact_m_aliens(nums, m, l, r)`: tổng lớn nhất khi chọn **đúng `m`** đoạn (với `1 ≤ m ≤ n // l`), dùng Aliens với `λ` có thể âm.

In [24]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def decode(key, M):
    # TODO: c = (-key) % M ; value = (key + c) // M ; trả (value, c)
    pass

# Bài 4 — TODO
def my_best_single(nums, l, r):
    # TODO: P tổng tiền tố; max P[i] - P[j] với i - j trong [l, r]
    pass

# Bài 5 — TODO
def exact_m_aliens(nums, m, l, r):
    # TODO: như aliens_key nhưng lo âm đủ nhỏ và bỏ qua nhánh f1 <= 0
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: `d_k = f(k) − f(k−1)`.
- Mức 2: `5, 2, 0, −5`.
- Mức 3: không tăng ⇒ lõm.

**Bài 2.**
- Mức 1: `f(k) − 3k` cho `k = 0..4`.
- Mức 2: `0, 2, 1, −2, −10`.
- Mức 3: `g(3) = 2` tại `k = 1`; số đoạn nhỏ nhất `1`.

**Bài 3.**
- Mức 1: `0 ≤ số_đoạn < M`.
- Mức 2: `key mod M` cho `(−số_đoạn) mod M`.
- Mức 3: `c = (-key) % M`.

**Bài 4.**
- Mức 1: `P[i] − P[j]` với `i − j ∈ [l, r]`.
- Mức 2: hai vòng `for`.
- Mức 3: `max(P[i] - P[i - ln] for ln in range(l, r+1) if i - ln >= 0)`.

**Bài 5.**
- Mức 1: tìm `λ` nguyên nhỏ nhất có `c_min(λ) ≤ m`, nay `λ` có thể âm.
- Mức 2: cận dưới `lo = −(tổng |nums|) − 1`, cận trên `hi = f(1) + 1`.
- Mức 3: trả `g(λ) + λ·m` (kể cả khi `λ ≤ 0`).

## 19. Đáp án tham khảo

In [25]:
# Bài 1
f = [0, 5, 7, 7, 2]
d = [f[k] - f[k - 1] for k in range(1, len(f))]
assert d == [5, 2, 0, -5] and all(d[i] >= d[i + 1] for i in range(len(d) - 1))
# Bài 2
sc = [f[k] - 3 * k for k in range(5)]
assert sc == [0, 2, 1, -2, -10] and max(sc) == 2 and sc.index(max(sc)) == 1

# Bài 3
def decode_ans(key, M):
    c = (-key) % M
    return (key + c) // M, c

for value in [-50, -1, 0, 3, 10**10]:
    for c in [0, 1, 7, 99]:
        assert decode_ans(value * 100 - c, 100) == (value, c), (value, c)
print("Bài 1–3 OK")

Bài 1–3 OK


In [26]:
# Bài 4
def my_best_single_ans(nums, l, r):
    n = len(nums)
    P = [0]
    for x in nums:
        P.append(P[-1] + x)
    best = None
    for i in range(1, n + 1):
        for ln in range(l, r + 1):
            if i - ln >= 0:
                v = P[i] - P[i - ln]
                if best is None or v > best:
                    best = v
    return best

for _ in range(200):
    n = rng.randint(1, 12)
    nums = [rng.randint(-9, 9) for _ in range(n)]
    l = rng.randint(1, n); r = rng.randint(l, n)
    P = [0]
    for x in nums: P.append(P[-1] + x)
    assert my_best_single_ans(nums, l, r) == best_single(P, n, l, r)

# Bài 5: đúng m đoạn (1 <= m <= n // l)
def exact_m_aliens_ans(nums, m, l, r):
    n = len(nums)
    P = [0] * (n + 1)
    for i, x in enumerate(nums):
        P[i + 1] = P[i] + x
    M_ = n + 2
    f1 = best_single(P, n, l, r)
    lo, hi = -(sum(abs(x) for x in nums) + 1) * 2, f1 + 1
    while lo < hi:
        mid = (lo + hi) // 2                       # // làm tròn xuống đúng cả với số âm
        if penalty_pass_key(P, n, l, r, mid, M_)[1] <= m:
            hi = mid
        else:
            lo = mid + 1
    g, _ = penalty_pass_key(P, n, l, r, lo, M_)
    return g + lo * m

def exact_values_ref(nums, l, r):
    return exact_values(nums, l, r)

rng = random.Random(42)
for _ in range(600):
    n = rng.randint(1, 10)
    nums = [rng.randint(-9, 9) for _ in range(n)]
    l = rng.randint(1, min(n, 3)); r = rng.randint(l, n)
    Kmax = n // l
    m = rng.randint(1, Kmax)
    assert exact_m_aliens_ans(nums, m, l, r) == exact_values(nums, l, r)[m], (nums, m, l, r)
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template Aliens trick (tối đa hóa, `f` lõm, phá hòa ít đoạn nhất):**

```python
f1 = đoạn đơn tốt nhất
if m == 1 or f1 <= 0: return f1                      # "ít nhất một đoạn"

def solve(lam):                                      # một lượt DP O(n), trả (g(lam), số đoạn nhỏ nhất)
    # key = giá_trị * M - số_đoạn ; thêm một đoạn: key_j - P[j]*M + P[i]*M - lam*M - 1
    ...
    c = (-key) % M ; return (key + c) // M, c

lo, hi = 0, f1 + 1                                   # λ nhỏ nhất có số_đoạn <= m
while lo < hi:
    mid = (lo + hi) // 2
    if solve(mid)[1] <= m: hi = mid
    else: lo = mid + 1
g, _ = solve(lo)
return g if lo == 0 else g + lo * m                  # nhớ cộng lại λ·m
```

**Checklist trước khi submit:**

- [ ] `f(k)` lõm — đã kiểm chứng với DP `O(n·m)` trên input nhỏ ngẫu nhiên.
- [ ] Phá hòa nhất quán (ít đoạn nhất) bằng khóa `giá_trị · M − số_đoạn`, `M = n + 2`.
- [ ] Nhị phân `λ ∈ [0, f(1) + 1]`, điều kiện `số_đoạn ≤ m` ⇒ `hi = mid`.
- [ ] Trả `g(λ) + λ·m` (và `g(0)` khi `λ = 0`).
- [ ] Xử lý `m = 1` hoặc `f(1) ≤ 0` ⇒ trả `f(1)` (ít nhất một đoạn).
- [ ] Cửa sổ `j ∈ [i − r, i − l]`, đoạn `[j, i)` nửa mở; deque lưu `h[j] − P[j]·M`.
- [ ] Hai đoạn được phép kề nhau.
- [ ] Đã đối chiếu vét cạn/DP `O(n·m)` trên input nhỏ.